In [ ]:
!pip install pgmpy

In [ ]:
import pandas as pd
import numpy as np
import math
from pgmpy.sampling import BayesianModelSampling
from pgmpy.estimators import BayesianEstimator
from pgmpy.factors.discrete import State
from pgmpy.inference import VariableElimination

from pgmpy.models import BayesianNetwork, DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import BeliefPropagation

In [ ]:
FILES_PATHS = ['https://github.com/fez2010/detection_de_ivresse_au_volant/raw/refs/heads/main/datasets/DonneesTestSymptomatiqueDRE_A2012.xlsx']

datasets = []
for path in FILES_PATHS:
  #get csv file encoding
  df = pd.read_excel(path,converters={'nr_nan':lambda x: pd.NA if x == '' else int(x)})

  df.columns = df.columns.str.capitalize()
  datasets.append(df)

print(datasets[0].shape)
datasets[0].head()

(116, 50)


,Testno,Nistagmus horizontal,Nistagmus vertical,Convergence,Pupilles,Stimulus,Paupieres,Ambiant,Noirceur,Direct,...,Sautillement,Depot pied,Alcool,Dépresseur,Stimulant,Hallucinogène,Anesthésique dissociatif,Narcotique analgésique,Inhalant,Canabis
0,1,0,0,1,0,1,0,4.0,8.0,4.5,...,1.0,1.0,0,0,1,0,0,0,0.0,0
1,2,0,0,1,0,1,0,6.5,6.5,4.5,...,0.0,1.0,1,0,0,0,0,0,0.0,1
2,3,1,0,1,0,1,0,7.5,7.0,4.5,...,0.0,1.0,1,0,0,0,0,0,0.0,1
3,4,1,0,1,0,1,0,5.5,8.0,4.5,...,0.0,1.0,1,0,0,0,0,0,0.0,1
4,5,1,1,1,0,1,1,3.0,6.5,2.0,...,0.0,1.0,0,1,0,0,0,0,0.0,1


In [ ]:
cibles = [
    'Alcool', 'Dépresseur', 'Stimulant', 'Hallucinogène',
    'Anesthésique dissociatif', 'Narcotique analgésique', 'Inhalant', 'Cannabis'
]

# Ordre strict des colonnes pour le tenseur PyTorch
colonnes_finales = [
    # --- BLOC 1 : Inférence Bayésienne (pgmpy sampler) ---
    # Probabilités issues de l'expertise DRE (Softmax-like)
    *[f'prob_{i}' for i in range(256)],

    # --- BLOC 2 : Biométrie & Vision (Capteurs) ---
    'perclos',      # Somnolence (0.0 - 1.0)
    'ataxia',       # Instabilité motrice (0.0 - 1.0)
    'gaze',         # Nystagmus / Fixation (0.0 - 1.0)

    # --- BLOC 3 : Pharmacocinétique (Expertise) ---
    'dt',           # Temps écoulé (Heures)
    'charge',       # Quantité résiduelle estimée (0.0 - 1.0)
    'i_crash',      # Intensité de la phase de descente (0.0 - 1.0)

    # --- BLOC 4 : Fiabilité (Flags de panne) ---
    'f_t',          # 1 si dt est mesuré, 0 si imputé par intervalle_regression
    'f_vis',        # Disponibilité caméra
    'f_imu',        # Disponibilité accéléromètre
    'f_gps',        # Disponibilité vitesse/trajectoire

    # --- BLOC 5 : Identité & Cible ---
    'profil_id',    # Bitmask normalisé (0.0 - 1.0)
    'target'        # Index de la classe réelle (0-7) -> Label d'entraînement
]

In [ ]:
import re

def clean_bif(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Remplacement des espaces dans les noms de variables/probabilités
    # Cherche les motifs 'variable Nom avec espace {' ou 'probability ( Nom avec espace |'
    content = re.sub(r'variable (.*?) \{', lambda m: 'variable ' + m.group(1).replace(' ', '_') + ' {', content)
    content = re.sub(r'probability \( (.*?) \)', lambda m: 'probability ( ' + m.group(1).replace(' ', '_') + ' )', content)

    # Suppression des accents spécifiques
    content = content.replace('Injecté', 'Injecte')
    content = content.replace('/', '_')

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(content)

In [ ]:
from pgmpy.readwrite import BIFReader

try:
    with open('model_final.bif', 'r') as f:
        content = f.read()
        print("Fichier lu avec succès, tentative de parsing...")

    reader = BIFReader('model_final.bif')
    model = reader.get_model()
    print("Parsing terminé.")
except Exception as e:
    try:

      print(f"Erreur détectée : {e}")
      # Utilisation
      clean_bif('model_final.bif', 'model_clean.bif')
      # Ensuite, essayez de lire le fichier nettoyé

      reader = BIFReader('model_clean.bif')
      model = reader.get_model()
      print("Parsing terminé.")
    except Exception as e:
      print(f"Erreur lors du parsing : {e}")
      import pickle

      # On suppose que 'model' est votre objet DiscreteBayesianNetwork
      #with open('model_final.pkl', 'rb') as fichier:
      #    model_final = pickle.load(fichier)

      #print("Modèle chargé !'")

Erreur détectée : [Errno 2] No such file or directory: 'model_final.bif'
Erreur lors du parsing : [Errno 2] No such file or directory: 'model_final.bif'


In [ ]:
def preparer_donnees_bayesian(df):
    # Nettoyage des noms de colonnes (espaces invisibles)
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.strip()

    # --- 1. SIGNES VITAUX (Imputation par la Médiane avant discrétisation) ---
    for col in ['Pouls', 'Température', 'Tension a', 'Tension b']:
        if col in df_clean.columns:
            # On remplit les vides par la valeur normale/médiane pour ne pas biaiser le réseau
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    df_clean['Pouls_cat'] = pd.cut(df_clean['Pouls'], bins=[0, 60, 90, 300], labels=['Bas', 'Normal', 'Eleve'])
    df_clean['Temp_cat'] = pd.cut(df_clean['Température'], bins=[0, 36.2, 37.5, 45], labels=['Bas', 'Normal', 'Eleve'])
    df_clean['Tension_a_cat'] = pd.cut(df_clean['Tension a'], bins=[0, 120, 140, 250], labels=['Bas', 'Normal', 'Eleve'])
    df_clean['Tension_b_cat'] = pd.cut(df_clean['Tension b'], bins=[0, 70, 90, 150], labels=['Bas', 'Normal', 'Eleve'])

    # --- 2. PUPILLES ET LUMIÈRE ---
    for col in ['Ambiant', 'Noirceur', 'Direct']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            df_clean[f'{col}_cat'] = pd.cut(df_clean[col], bins=[0, 3.0, 5.0, 15], labels=['Myosis', 'Normal', 'Mydriase'])

    # Gestion spécifique de 'Dilatation bond' (souvent binaire ou manquante)
    if 'Dilatation bond' in df_clean.columns:
        df_clean['Dilatation bond_cat'] = df_clean['Dilatation bond'].fillna(0).apply(lambda x: 1 if x > 0 else 0)
    if 'Reaction lum' in df_clean.columns:
        df_clean['Reaction lum_cat'] = df_clean['Reaction lum'].fillna(0).apply(lambda x: 1 if x > 0 else 0)
    if 'Talon/orteille' in df_clean.columns:
        df_clean['Talon/orteille_cat'] = df_clean['Talon/orteille'].fillna(0).apply(lambda x: 1 if x > 0 else 0)
    if 'Nistagmus vertical' in df_clean.columns:
        df_clean['Nistagmus vertical_cat'] = df_clean['Nistagmus vertical'].fillna(0).apply(lambda x: 1 if x > 0 else 0)
    if 'Nistagmus horizontal' in df_clean.columns:
        df_clean['Nistagmus horizontal_cat'] = df_clean['Nistagmus horizontal'].fillna(0).apply(lambda x: 1 if x > 0 else 0)

    # --- 3. TESTS PSYCHOMOTEURS ---
    # Romberg : Si vide, on considère 'Echec' par prudence ou 'Inconnu'
    for col in ['Romberg 1', 'Romberg 2', 'Romberg 3']:
        if col in df_clean.columns:
            # On traite les NaN comme un échec au test (incapacité à le faire)
            df_clean[f'{col}_cat'] = df_clean[col].apply(lambda x: 'Echec' if (pd.isna(x) or abs(x - 30) > 5) else 'Reussi')

    for col in ['Equi g', 'Equi d']:
        if col in df_clean.columns:
            # Remplissage par 0 (instabilité totale) si vide
            df_clean[f'{col}_cat'] = pd.cut(df_clean[col].fillna(0), bins=[-1, 25, 31], labels=['Instable', 'Stable'])

    # Pas et erreurs de marche : NaN = 1 (on assume l'erreur si non mentionné "réussi")
    cols_erreurs = ['Arret marche', 'Pas hors ligne', 'Utilisation bras', 'Sautillement', 'Depot pied', 'Balancement']
    for col in cols_erreurs:
        if col in df_clean.columns:
            df_clean[f'{col}_cat'] = df_clean[col].fillna(1).apply(lambda x: 1 if x > 0 else 0)

    # --- 4. VARIABLES BINAIRES & CIBLES (NaN = 0) ---
    binaires_explicatives = [
         'Convergence', 'Pupilles',
        'Stimulus', 'Paupieres', 'Injection', 'Tonus musculaire', 'Yeux normaux',
        'Injecté sang', 'Conjonctive rouge', 'Larmoyant', 'Test toucher',
        'Perte equilibre', 'Depart hatif', 'Pivot incorrect'
    ]

    cibles = ['Alcool', 'Dépresseur', 'Stimulant', 'Hallucinogène',
              'Anesthésique dissociatif', 'Narcotique analgésique', 'Inhalant', 'Cannabis']

    for col in binaires_explicatives + cibles:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(0).astype(int)

    # --- 5. PROFIL_ID (Doit être calculé sans NaN) ---
    def generer_id_profil(row):
        valeur = 0
        for i, cible in enumerate(cibles):
            if row.get(cible, 0) == 1:
                valeur += (2**i)
        return valeur

    df_clean['Profil_ID'] = df_clean.apply(generer_id_profil, axis=1)

    # --- 6. SÉLECTION FINALE (Vérification de l'existence des colonnes) ---
    colonnes_base = ['Profil_ID'] + [c for c in cibles if c in df_clean.columns]
    colonnes_cats = [c for c in df_clean.columns if '_cat' in c]
    colonnes_bins = [c for c in binaires_explicatives if c in df_clean.columns]

    return df_clean[colonnes_base + colonnes_cats + colonnes_bins]

In [ ]:
def obtenir_intervalle_regression(categorie):
    """
    Retourne l'intervalle de début de régression sous forme de tableau [min, max] (en heures).
    """
    # Valeurs normalisées en heures
    mapping = {
        "Alcool": {
            "intervalle": [0.5, 4],
            "symptomes": "Déshydratation, anxiété ('hangxiety')",
            "risque": "Rebond d'excitabilité"
        },
        "Dépresseur": {
            "intervalle": [4, 24],
            "symptomes": "Insomnie, tremblements, hypersensibilité",
            "risque": "Sevrage physique"
        },
        "Stimulant": {
            "intervalle": [1, 6],
            "symptomes": "Épuisement, anhédonie, paranoïa",
            "risque": "Idées noires, envie forte de reconsommer"
        },
        "Hallucinogène": {
            "intervalle": [6, 12],
            "symptomes": "Fatigue mentale, confusion",
            "risque": "Flashbacks ou dépersonnalisation"
        },
        "Anesthésique dissociatif": {
            "intervalle": [1, 2],
            "symptomes": "Désorientation, nausées, vertiges",
            "risque": "Chutes et accidents physiques"
        },
        "Narcotique analgésique": {
            "intervalle": [3, 8],
            "symptomes": "Douleurs diffuses, irritabilité",
            "risque": "Retour violent de la douleur"
        },
        "Inhalant": {
            "intervalle": [0.1, 0.5], # 6 à 30 minutes
            "symptomes": "Maux de tête, somnolence",
            "risque": "Arythmie cardiaque"
        },
        "Cannabis": {
            "intervalle": [3, 48],
            "symptomes": "Léthargie ('fog'), apathie",
            "risque": "Baisse de la motivation"
        }
    }

    cat_clean = categorie.strip()
    return mapping.get(cat_clean, "Catégorie non trouvée.")

# Exemple d'utilisation pour une application :
# intervalle = obtenir_intervalle_regression("stimulant")["intervalle"]
# print(f"Début critique entre {intervalle[0]}h et {intervalle[1]}h")

In [ ]:
def calculer_intensite_crash(categorie, delta_t):
    data = obtenir_intervalle_regression(categorie)
    if isinstance(data, str): return 0.0

    t_min, t_max = data["intervalle"]

    # Si on est dans la fenêtre de crash
    if t_min <= delta_t <= t_max:
        # Calcul d'une courbe en cloche (Gaussienne simplifiée)
        # L'intensité est maximale au centre de l'intervalle
        centre = (t_min + t_max) / 2
        largeur = (t_max - t_min) / 2
        intensite = math.exp(-((delta_t - centre)**2) / (2 * (largeur/2)**2))
        return round(intensite, 4)

    return 0.0

In [ ]:
import math

def simulateur_expert(categorie, dose, temps, profil="occasionnel"):
    """
    Simulateur de régression plasmatique.
    profil: 'occasionnel' ou 'chronique' (principalement pour le cannabis)
    """
    cat = categorie

    # Paramètres de demi-vie (en heures)
    # k = ln(2) / t_1/2
    config = {
        "Alcool": {"mode": "ordre_0", "k": 0.15}, # g/L par heure
        "Dépresseur": {"mode": "ordre_1", "t12": 24},
        "Stimulant": {"mode": "ordre_1", "t12": 10},
        "Hallucinogène": {"mode": "ordre_1", "t12": 4},
        "Anesthésique dissociatif": {"mode": "ordre_1", "t12": 3},
        "Narcotique analgésique": {"mode": "ordre_1", "t12": 4},
        "Inhalant": {"mode": "ordre_1", "t12": 0.5},
        "Cannabis": {
            "mode": "ordre_1",
            "t12": 32 if profil == "occasionnel" else 240 # 240h = 10 jours
        }
    }

    if cat not in config:
        return "Catégorie non reconnue."

    c = config[cat]

    if c["mode"] == "ordre_0":
        reste = max(0, dose - (c["k"] * temps))
    else:
        k = 0.693 / c["t12"]
        reste = dose * math.exp(-k * temps)

    return {
        "Substance": categorie,
        "Profil": profil if cat == "Cannabis" else "standard",
        "Reste": round(reste, 3),
        "Elimination": "99% éliminé" if reste < (dose * 0.01) else "En cours"
    }

# Test pour un utilisateur chronique de cannabis après 48h
# print(simulateur_expert("Cannabis", 100, 48, profil="chronique"))

In [ ]:
import math

class SimulateurPharmacocinetique:
    def __init__(self):
        # Configuration technique des catégories
        # intervalle_crash: [min, max] en heures
        # t12: demi-vie moyenne en heures
        # mode: 0 (linéaire/alcool), 1 (exponentiel/autres)
        self.data = {
            "Alcool": {"mode": 0, "k": 0.15, "t12": 4, "intervalle_crash": [0.5, 4],
                       "symptomes": "Hangxiety, déshydratation", "risque": "Rebond d'excitabilité"},
            "Dépresseur": {"mode": 1, "t12": 24, "intervalle_crash": [4, 24],
                          "symptomes": "Tremblements, insomnie", "risque": "Sevrage physique"},
            "Stimulant": {"mode": 1, "t12": 2, "intervalle_crash": [1, 6],
                         "symptomes": "Épuisement, paranoïa", "risque": "Idées noires (Crash)"},
            "Hallucinogène": {"mode": 1, "t12": 5, "intervalle_crash": [6, 12],
                             "symptomes": "Confusion, fatigue mentale", "risque": "Flashbacks"},
            "Anesthésique dissociatif": {"mode": 1, "t12": 3, "intervalle_crash": [1, 2],
                                  "symptomes": "Vertiges, nausées", "risque": "Chutes physiques"},
            "Narcotique analgésique": {"mode": 1, "t12": 4, "intervalle_crash": [3, 8],
                                "symptomes": "Douleurs, irritabilité", "risque": "Retour douleur"},
            "Inhalant": {"mode": 1, "t12": 0.5, "intervalle_crash": [0.1, 0.5],
                        "symptomes": "Maux de tête, léthargie", "risque": "Arythmie"},
            "Cannabis": {"mode": 1, "t12_occ": 32, "t12_chr": 240, "intervalle_crash": [3, 48],
                         "symptomes": "Brouillard mental, apathie", "risque": "Amotivation"}
        }

    def _get_k(self, cat, profil="occasionnel"):
        conf = self.data[cat]
        if cat == "Cannabis":
            t12 = conf["t12_chr"] if profil == "chronique" else conf["t12_occ"]
            return 0.693 / t12
        return 0.693 / conf["t12"] if conf["mode"] == 1 else conf["k"]

    def calculer_etat(self, cat, dose_initiale, heures_passees, profil="occasionnel"):
        if cat not in self.data: return "Catégorie inconnue"

        conf = self.data[cat]
        k = self._get_k(cat, profil)

        # 1. Calcul de la régression (Quantité restante)
        if conf["mode"] == 0:
            reste = max(0, dose_initiale - (k * heures_passees))
        else:
            reste = dose_initiale * math.exp(-k * heures_passees)

        # 2. Prédiction du temps restant (jusqu'à 1% de la dose)
        seuil = dose_initiale * 0.01
        if reste <= seuil:
            t_restant = 0
        else:
            t_restant = (reste / k) if conf["mode"] == 0 else (math.log(seuil / reste) / -k)

        return {
            "substance": cat,
            "quantite_actuelle": round(reste, 3),
            "unite": "g/L" if cat == "Alcool" else "mg/u",
            "temps_avant_elimination_h": round(t_restant, 2),
            "debut_crash_intervalle": conf["intervalle_crash"],
            "symptomes_prevus": conf["symptomes"],
            "alerte_securite": conf["risque"]
        }

    def evaluer_combinaison(self, cat1, cat2):
        # Logique simplifiée d'interaction
        types = {"Alcool": "D", "Dépresseur": "D", "Narcotique analgésique": "D", "Stimulant": "S"}
        t1, t2 = types.get(cat1, "O"), types.get(cat2, "O")

        if t1 == t2 == "D": return "⚠️ DANGER CRITIQUE : Synergie respiratoire (1+1=5)"
        if (t1 == "S" and t2 == "D") or (t1 == "D" and t2 == "S"):
            return "⚠️ RISQUE ÉLEVÉ : Effet masquant (Surdose invisible)"
        return "ℹ️ Interaction possible : Métabolisme ralenti"

# --- TEST DU CODE ---
sim = SimulateurPharmacocinetique()
print(sim.calculer_etat("Stimulant", 100, 4)) # 100mg après 4h
print(sim.evaluer_combinaison("Alcool", "Narcotique analgésique"))

{'substance': 'Stimulant', 'quantite_actuelle': 25.007, 'unite': 'mg/u', 'temps_avant_elimination_h': 9.29, 'debut_crash_intervalle': [1, 6], 'symptomes_prevus': 'Épuisement, paranoïa', 'alerte_securite': 'Idées noires (Crash)'}
⚠️ DANGER CRITIQUE : Synergie respiratoire (1+1=5)


In [ ]:
import math

def predire_temps_restant(categorie, dose_actuelle, seuil_securite=0.01):
    """
    Calcule le temps (en heures) pour passer de dose_actuelle au seuil_securite.
    """
    cat = categorie.strip()

    # Données techniques : [Type de régression, Constante k ou t1/2]
    # Pour Ordre 1, k = 0.693 / t12
    config = {
        "Alcool": {"mode": 0, "k": 0.15},
        "Dépresseur": {"mode": 1, "k": 0.693 / 24},
        "Stimulant": {"mode": 1, "k": 0.693 / 2},
        "Hallucinogène": {"mode": 1, "k": 0.693 / 5},
        "Anesthésique dissociatif": {"mode": 1, "k": 0.693 / 3},
        "Narcotique analgésique": {"mode": 1, "k": 0.693 / 4},
        "Inhalant": {"mode": 1, "k": 0.693 / 0.5},
        "Cannabis": {"mode": 1, "k": 0.693 / 32} # Profil occasionnel par défaut
    }

    if cat not in config: return "Catégorie non reconnue."

    c = config[cat]

    if dose_actuelle <= seuil_securite:
        return 0.0

    if c["mode"] == 0:
        # Temps = Dose / Taux d'élimination
        temps_restant = dose_actuelle / c["k"]
    else:
        # Temps = ln(C_final / C_initial) / -k
        # Ici on veut atteindre le seuil_securite
        temps_restant = math.log(seuil_securite / dose_actuelle) / -c["k"]

    return round(temps_restant, 2)

# Exemple : Combien de temps pour éliminer 0.8g/L d'alcool ?
# print(f"Temps restant : {predire_temps_restant('alcool', 0.8)} heures")

In [ ]:
def transformer_samples_en_probas(df_samples, categories_liste):
    """
    Convertit les états binaires du PGM en vecteurs de probabilités (18 cols).
    """
    n_samples = len(df_samples)
    n_categories = 256  # Votre structure de sortie fixe

    # Matrice de probabilités finale
    probs_matrix = np.zeros((n_samples, n_categories))

    for i, (idx, row) in enumerate(df_samples.iterrows()):
        # 1. Initialisation avec un bruit de fond (prior faible)
        # Dirichlet assure que la somme = 1 et que les valeurs sont > 0
        v = np.random.dirichlet(np.ones(n_categories) * 0.05)

        # 2. Récupération de l'ID de profil de l'échantillon
        p_id = int(row.get('Profil_ID', 0))

        # 3. Attribution des scores basés sur le Bitmask
        # Pour chaque bit actif dans le Profil_ID, on booste la probabilité correspondante
        for j in range(len(categories_liste)):
            if (p_id >> j) & 1:
                # La substance est présente : on lui donne un score dominant (0.6 - 0.9)
                v[j] += np.random.uniform(0.6, 0.9)

        # 4. Normalisation finale (Softmax-like)
        # Cela garantit que la somme des 18 colonnes est exactement 1.0
        probs_matrix[i] = v / v.sum()

    return probs_matrix

In [ ]:
def transformer_sample_en_probas_padded(row):
    """
    Convertit les états binaires du PGM en vecteurs de probabilités (256 cols).
    """

    n_categories = 256  # Votre structure de sortie fixe

    # Matrice de probabilités finale
    probs_vector = np.zeros(n_categories)

    v = np.zeros(n_categories)
    # 2. Récupération de l'ID de profil de l'échantillon
    for j in range(len(row.values)):
      p_id = int(row.state_names['Profil_ID'][j])
      probs_vector[p_id] = row.values[j]

    return probs_vector
def transformer_samples_en_probas_padded(df_samples):
    """
    Convertit les états binaires du PGM en vecteurs de probabilités (256 cols).
    """
    n_samples = len(df_samples)
    n_categories = 256  # Votre structure de sortie fixe

    # Matrice de probabilités finale
    probs_matrix = np.zeros((n_samples, n_categories))

    for i, (idx, row) in enumerate(df_samples.iterrows()):
        probs_matrix[i] = transformer_sample_en_probas_padded(row)

    return probs_matrix

In [ ]:
def decoder_profil(id_profil, liste_cibles):
    """
    Transforme un ID (0-255) en dictionnaire de présence/absence.
    """
    resultat = {}
    for i, cible in enumerate(liste_cibles):
        # Vérifie si le bit i est actif dans l'ID
        est_present = bool(id_profil & (1 << i))
        resultat[cible] = est_present
    return resultat
def diagnostic_expert(model, donnees_observation):
    """
    donnees_observation : dict des symptômes (ex: {'Pouls_cat': 'Eleve', 'Nistagmus': 1})
    """
    inference = VariableElimination(model)

    # Calcul de la distribution de probabilité sur Profil_ID
    query_result = inference.query(variables=['Profil_ID'], evidence=donnees_observation)
    print(query_result)
    # Extraction des 3 meilleures probabilités
    indices = np.argsort(query_result.values)[::-1][:3]

    print("--- RÉSULTATS DU RÉSEAU BAYÉSIEN DISCRET ---")
    for idx in indices:
        prob = query_result.values[idx]
        if prob > 0.01:
            # On utilise le décodeur bitwise (vu précédemment) pour lire les drogues
            combinaison = decoder_profil(idx, cibles)
            noms = [n for n, p in combinaison.items() if p]
            label = ", ".join(noms) if noms else "Négatif (Sain)"
            print(f"Probabilité {prob:.1%} : [{label}]")

# Exemple d'appel :


In [ ]:
def generer_dataset_final_256(model_bn, n_par_profil=10):
    sampler = BayesianModelSampling(model_bn)
    data = []
    categories_liste = [cat for cat in cibles]
    # 1. Boucle sur les 256 combinaisons de drogues (00000000 à 11111111)
    for p_id in range(256):
        # A. Inférence via Treillis (Sampling sous évidence Profil_ID)
        try:
            evidence = [State('Profil_ID', p_id)]
            # On génère n échantillons pour ce profil spécifique
            samples = sampler.likelihood_weighted_sample(evidence=evidence, size=n_par_profil)

            # B. Transformation en Probabilités (18 colonnes)
            # Utilise votre fonction de mapping pour créer les scores continus
            probs_pgm = transformer_samples_en_probas(samples,categories_liste)

            # C. Analyse de la cinétique du profil
            # On identifie les bits actifs pour trouver la drogue dominante
            bits_actifs = [i for i in range(8) if (p_id >> i) & 1]

            if not bits_actifs: # Cas "Sain" (p_id = 0)
                cat_ref = "Alcool"
                t_min, t_max = 0.1, 0.5
                charge_base = 0.0
            else:
                # On prend la drogue la plus lente (max t_max) comme référence temporelle
                intervals = [obtenir_intervalle_regression(categories_liste[b])["intervalle"] for b in bits_actifs]
                t_min = min([i[0] for i in intervals])
                t_max = max([i[1] for i in intervals])
                cat_ref = categories_liste[bits_actifs[0]]
                charge_base = 100.0

            for j in range(n_par_profil):
                # Tirage de dt dans la fenêtre de l'expertise
                dt = np.random.uniform(t_min * 0.5, t_max * 1.2)

                # Calcul de l'état métabolique
                est_chr = "chronique" if p_id >= 128 else "occasionnel"
                etat = sim.calculer_etat(cat_ref, charge_base, dt, profil=est_chr)

                charge = etat["quantite_actuelle"] / 100.0
                i_crash = calculer_intensite_crash(cat_ref, dt)

                # D. Biométrie (Logique : Ataxie liée à la charge, Perclos lié au crash)
                perclos = np.clip(0.05 + (0.35 * i_crash) + np.random.normal(0, 0.02), 0, 1)
                ataxia = np.clip(0.02 + (0.25 * charge) + np.random.normal(0, 0.02), 0, 1)

                # E. Construction du vecteur final (28 colonnes)
                target_idx = bits_actifs[0] if bits_actifs else 8 # 8 = Classe 'Sain'

                row = list(probs_pgm[j]) + [
                    perclos, ataxia, 0.2,            # Bio (3)
                    dt, charge, i_crash,             # Cinétique (3)
                    1.0, 1.0, 1.0, 1.0,              # Flags (4)
                    p_id / 255.0,                    # Profil_ID Norm (1)
                    float(target_idx)                # Target (1)
                ]
                data.append(row)

        except Exception as e:
            print(f"Erreur Profil {p_id}: {e}")
            continue

    return pd.DataFrame(data, columns=colonnes_finales)

In [ ]:
# Utilisation :
df_bayesian = preparer_donnees_bayesian(df)

In [ ]:
# 1. SYMPTÔMES OCULAIRES (Mélange de binaires et catégorisés)
oculaires = [
    'Nistagmus horizontal_cat', 'Nistagmus vertical_cat', 'Convergence',
    'Pupilles', 'Stimulus', 'Paupieres', 'Ambiant_cat',
    'Noirceur_cat', 'Direct_cat', 'Dilatation bond_cat',
    'Reaction lum_cat', 'Yeux normaux', 'Injecté sang',
    'Conjonctive rouge', 'Larmoyant'
]

# 2. SIGNES VITAUX (Tous catégorisés par pd.cut)
vitaux = [
    'Pouls_cat', 'Temp_cat', 'Tension_a_cat', 'Tension_b_cat',
    'Tonus musculaire', 'Injection'
]

# 3. TESTS PSYCHOMOTEURS (Performance et Temps)
tests_mouvements = [
    'Romberg 1_cat', 'Romberg 2_cat', 'Romberg 3_cat',
    'Test toucher', 'Perte equilibre', 'Depart hatif',
    'Pivot incorrect', 'Equi g_cat', 'Equi d_cat'
]

# 4. ERREURS DE MARCHE (Binaires : 0 ou 1)
tests_erreurs = [
    'Arret marche_cat', 'Pas hors ligne_cat', 'Utilisation bras_cat',
    'Sautillement_cat', 'Depot pied_cat', 'Balancement_cat', 'Talon/orteille_cat'
]
# 2. Création de la liste des arcs (Profil_ID -> Symptôme)
toutes_explicatives = oculaires + vitaux + tests_mouvements + tests_erreurs

In [ ]:
# 1. Définition des 256 états possibles (0 à 255)
etats_complets = list(range(256))

# 2. Création de la CPD pour Profil_ID (Equiprobable pour le sampling)
cpd_profil = TabularCPD(
    variable='Profil_ID',
    variable_card=256,
    values=[[1/256]] * 256,
    state_names={'Profil_ID': etats_complets}
)



In [ ]:
def generer_cpd_symptome(nom_symptome, parent_name, prob_dict):
    """
    Crée une CPD où la probabilité dépend des bits actifs dans le Profil_ID.
    prob_dict: {index_drogue: probabilité_si_présent}
    """
    table_vraie = [] # Probabilité que le symptôme soit présent (1)

    for p_id in range(256):
        # On calcule la probabilité combinée (Synergie)
        # Logique simple : on prend le max des probabilités des drogues présentes
        p_combinee = 0.05 # Bruit de base (faux positif)

        for idx_drogue, p_presence in prob_dict.items():
            if (p_id >> idx_drogue) & 1:
                p_combinee = max(p_combinee, p_presence)

        table_vraie.append(p_combinee)

    # La CPD doit contenir [P(0), P(1)] pour chaque état du parent
    values = [
        [1 - p for p in table_vraie], # Probabilité Symptôme = 0
        table_vraie                   # Probabilité Symptôme = 1
    ]

    return TabularCPD(
        variable=nom_symptome,
        variable_card=2,
        values=values,
        evidence=[parent_name],
        evidence_card=[256],
        state_names={nom_symptome: [0, 1], parent_name: list(range(256))}
    )

# Exemple pour l'Ataxie (influencée par Alcool(0) et Cannabis(7))
cpd_ataxia = generer_cpd_symptome('Ataxia', 'Profil_ID', {0: 0.8, 7: 0.4})

In [ ]:
model_final = DiscreteBayesianNetwork()
# Lister tous vos signes cliniques (ceux du dictionnaire prob_experts)
# signes_cliniques = list(prob_experts.keys())

model_final.add_nodes_from(['Profil_ID']  + toutes_explicatives)
model_final.add_edges_from([('Profil_ID', s) for s in  toutes_explicatives])

# Ajout de la CPD Profil_ID (déjà définie à 256 états)
model_final.add_cpds(cpd_profil)

# Entraînement robuste
model_final.fit(
    df_bayesian,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

# Génération et ajout des 18 CPD de signes
#for symptome, config in prob_experts.items():
#    cpd = generer_cpd_symptome(symptome, 'Profil_ID', config)
#    model_final.add_cpds(cpd)

# Vérification de la validité du modèle
if model_final.check_model():
    print("✅ Modèle Bayésien validé avec 256 états et 18 signes.")

# Initialisation du moteur d'inférence par Treillis
inference_jt = BeliefPropagation(model_final)

✅ Modèle Bayésien validé avec 256 états et 18 signes.


In [ ]:
diagnostic_expert(model_final,{'Nistagmus horizontal_cat': 1, 'Pouls_cat': 'Normal', 'Convergence': 1})

+---------------+------------------+
| Profil_ID     |   phi(Profil_ID) |
+===============+==================+
| Profil_ID(0)  |           0.0521 |
+---------------+------------------+
| Profil_ID(1)  |           0.1210 |
+---------------+------------------+
| Profil_ID(2)  |           0.4129 |
+---------------+------------------+
| Profil_ID(3)  |           0.1805 |
+---------------+------------------+
| Profil_ID(4)  |           0.0102 |
+---------------+------------------+
| Profil_ID(5)  |           0.0111 |
+---------------+------------------+
| Profil_ID(6)  |           0.1414 |
+---------------+------------------+
| Profil_ID(7)  |           0.0130 |
+---------------+------------------+
| Profil_ID(8)  |           0.0010 |
+---------------+------------------+
| Profil_ID(9)  |           0.0024 |
+---------------+------------------+
| Profil_ID(16) |           0.0018 |
+---------------+------------------+
| Profil_ID(19) |           0.0191 |
+---------------+------------------+
|

In [ ]:
try:
    # Test ciblé sur le profil problématique
    test_131 = inference_jt.query(variables=['Nistagmus horizontal_cat'], evidence={'Profil_ID': 131})
    print("Test Profil 131 réussi !",test_131.variables)
    print(f"Proba Nistagmus horizontal_cat (état 1) : {test_131.values[1]:.2%}")
except Exception as e:
    print(f"Erreur persistante sur le Profil 131 : {e}")

Erreur persistante sur le Profil 131 : 'state: 131 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'


In [ ]:
dataset_augmented = generer_dataset_final_256(model_final,500)

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

...

Erreur Profil 33: 'state: 33 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 34: 'state: 34 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 35: 'state: 35 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'


  0%|          | 0/38 [00:00<?, ?it/s]

Erreur Profil 37: 'state: 37 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 38: 'state: 38 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 39: 'state: 39 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 40: 'state: 40 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3

  0%|          | 0/38 [00:00<?, ?it/s]

Erreur Profil 67: 'state: 67 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 68: 'state: 68 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 69: 'state: 69 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(16), np.int64(19), np.int64(32), np.int64(36), np.int64(66)]'
Erreur Profil 70: 'state: 70 is an unknown for variable: Profil_ID. It must be one of [np.int64(0), np.int64(1), np.int64(2), np.int64(3

In [ ]:
dataset_augmented

,prob_0,prob_1,prob_2,prob_3,prob_4,prob_5,prob_6,prob_7,prob_8,prob_9,...,gaze,dt,charge,i_crash,f_t,f_vis,f_imu,f_gps,profil_id,target
0,4.964276e-21,1.708396e-02,2.722865e-07,1.174000e-04,7.741993e-17,7.743116e-05,3.112941e-06,8.026683e-12,4.545348e-05,4.989522e-06,...,0.2,0.063047,0.00000,0.0000,1.0,1.0,1.0,1.0,0.000000,8.0
1,9.006932e-04,8.167454e-06,4.409943e-10,1.361240e-27,9.004456e-04,2.632428e-04,4.350324e-16,4.350576e-17,1.535390e-23,2.611635e-06,...,0.2,0.337424,0.00000,0.0000,1.0,1.0,1.0,1.0,0.000000,8.0
2,9.054268e-02,1.188450e-12,1.936465e-05,6.442054e-06,6.712424e-06,7.290902e-10,2.920514e-09,8.477941e-06,3.307871e-08,6.411259e-12,...,0.2,0.226960,0.00000,0.0000,1.0,1.0,1.0,1.0,0.000000,8.0
3,6.283511e-11,2.369757e-13,1.774139e-10,1.704987e-03,2.215119e-01,2.559631e-03,8.245699e-03,3.889998e-03,7.475666e-08,1.795000e-06,...,0.2,0.171769,0.00000,0.0000,1.0,1.0,1.0,1.0,0.000000,8.0
4,4.161353e-22,1.852237e-31,2.751414e-09,3.705682e-09,4.059964e-09,1.005474e-08,6.500964e-04,8.295508e-07,9.643555e-07,4.633214e-11,...,0.2,0.441989,0.00000,0.0000,1.0,1.0,1.0,1.0,0.000000,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7495,7.453871e-04,2.782296e-01,1.610591e-10,2.677992e-13,5.416268e-05,2.709871e-06,2.990640e-01,6.565417e-18,3.832951e-12,7.228365e-03,...,0.2,12.289657,0.70127,0.9432,1.0,1.0,1.0,1.0,0.258824,1.0
7496,4.484171e-06,2.968264e-01,4.250528e-07,8.738135e-20,2.744497e-04,6.277761e-09,2.819973e-01,3.625093e-49,2.007186e-36,1.736942e-06,...,0.2,22.350015,0.52448,0.2480,1.0,1.0,1.0,1.0,0.258824,1.0
7497,4.054025e-04,2.828823e-01,2.522618e-11,2.866263e-05,3.103679e-11,5.246216e-25,3.095284e-01,3.397188e-23,2.993892e-15,2.910376e-04,...,0.2,12.230320,0.70247,0.9393,1.0,1.0,1.0,1.0,0.258824,1.0
7498,2.323248e-09,2.539056e-01,2.292790e-14,4.742940e-09,5.243150e-03,8.319854e-22,3.487542e-01,6.944389e-05,1.844599e-20,2.256331e-05,...,0.2,26.782360,0.46147,0.0000,1.0,1.0,1.0,1.0,0.258824,1.0


In [ ]:
def generer_id_profil(row):
    """
    Transforme les 8 colonnes binaires en un entier unique (0-255).
    Ex: Alcool seul = 1, Alcool + Canabis = 129, etc.
    """
    valeur = 0
    for i, cible in enumerate(cibles):
        if row[cible] == 1:
            valeur += (2**i)
    return valeur

In [ ]:
import pandas as pd
import numpy as np

def ajuster_et_fusionner_final_expert(df_reel, df_genere, categories_liste):
    """
    df_reel : Vos 116 lignes réelles
    df_genere : Vos 2560 lignes issues du Treillis (PGM)
    categories_liste : La liste ordonnée ['Alcool', 'Dépresseur', ..., 'Cannabis']
    """
    # 1. Définition des colonnes cibles pour le MLP (28 colonnes au total)
    cols_probas = [f'prob_{i}' for i in range(256)]
    cols_bio_kin = ['perclos', 'ataxia', 'gaze', 'dt', 'charge', 'i_crash']
    cols_flags = ['f_t', 'f_vis', 'f_imu', 'f_gps']
    colonnes_finales = cols_probas + cols_bio_kin + cols_flags + ['profil_id', 'target']

    df_r = df_reel.copy()

    # S'assurer que les colonnes de probabilités existent dans le réel (Padding initial)
    for c in cols_probas:
        if c not in df_r.columns:
            df_r[c] = 0.5  # Padding neutre (Incertitude 50%)

    # 2. Boucle d'ajustement pour chaque ligne du dataset réel
    for idx, row in df_r.iterrows():
        p_id = int(row.get('Profil_ID', 0))

        # Décomposition du bitmask pour identifier les drogues présentes
        # (L'erreur venait d'ici : on force b à être un index entier pour la liste)
        bits_actifs = [b for b in range(8) if (p_id >> b) & 1]

        if not bits_actifs: # Cas "Sain"
            cat_ref = "Alcool"
            t_min, t_max = 0.1, 0.5
            charge_base = 0.0
        else:
            # Récupération sécurisée des intervalles de régression
            try:
                intervals = []
                for b in bits_actifs:
                    nom_drogue = categories_liste[b] # Accès par index entier
                    info = obtenir_intervalle_regression(nom_drogue)
                    intervals.append(info["intervalle"])

                t_min = min([i[0] for i in intervals])
                t_max = max([i[1] for i in intervals])
                cat_ref = categories_liste[bits_actifs[0]] # Drogue dominante
                charge_base = 100.0
            except (IndexError, KeyError):
                cat_ref = "Alcool"
                t_min, t_max = 1.0, 4.0
                charge_base = 100.0

        # --- B. GESTION DU TEMPS (dt) ET PADDING ---
        dt_val = row.get('dt', 0)
        if pd.isna(dt_val) or dt_val <= 0:
            # On simule un temps réaliste pour combler le vide (Padding Cinétique)
            dt_val = np.random.uniform(t_min, t_max)
            df_r.at[idx, 'f_t'] = 0.0  # Indique que le temps est estimé
        else:
            df_r.at[idx, 'f_t'] = 1.0  # Temps réel fiable

        # --- C. CALCUL DES ÉTATS MÉTABOLIQUES ---
        est_chr = "chronique" if (p_id & 128) else "occasionnel"
        # Appel à votre simulateur cinétique
        etat = sim.calculer_etat(cat_ref, charge_base, dt_val, profil=est_chr)

        # Mise à jour des valeurs calculées
        df_r.at[idx, 'dt'] = dt_val
        df_r.at[idx, 'charge'] = etat['quantite_actuelle'] / 100.0
        df_r.at[idx, 'i_crash'] = calculer_intensite_crash(cat_ref, dt_val)
        df_r.at[idx, 'profil_id'] = p_id

        # Target : Index de la première drogue trouvée (ou 8 si sain)
        df_r.at[idx, 'target'] = bits_actifs[0] if bits_actifs else 8

    # 3. Alignement des structures et Fusion
    # On force le même ordre de colonnes pour les deux DataFrames
    df_r_final = df_r.reindex(columns=colonnes_finales, fill_value=0.0)
    df_g_final = df_genere.reindex(columns=colonnes_finales, fill_value=0.0)

    dataset_total = pd.concat([df_r_final, df_g_final], ignore_index=True)

    # 4. Normalisation finale du Profil_ID pour le MLP
    dataset_total['profil_id'] = dataset_total['profil_id'] / 255.0

    return dataset_total.sample(frac=1).reset_index(drop=True)

In [ ]:
# 1. Lire les données

df_bayesian

,Profil_ID,Alcool,Dépresseur,Stimulant,Hallucinogène,Anesthésique dissociatif,Narcotique analgésique,Inhalant,Pouls_cat,Temp_cat,...,Injection,Tonus musculaire,Yeux normaux,Injecté sang,Conjonctive rouge,Larmoyant,Test toucher,Perte equilibre,Depart hatif,Pivot incorrect
0,4,0,0,1,0,0,0,0,Eleve,Normal,...,0,0,0,1,0,0,0,1,1,1
1,1,1,0,0,0,0,0,0,Eleve,Normal,...,0,0,0,1,0,0,4,1,0,1
2,1,1,0,0,0,0,0,0,Eleve,Normal,...,0,0,0,0,1,0,4,0,0,1
3,1,1,0,0,0,0,0,0,Eleve,Normal,...,0,0,0,0,1,0,5,0,0,1
4,2,0,1,0,0,0,0,0,Normal,Normal,...,0,0,1,0,0,0,5,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,0,0,0,0,0,0,0,0,Normal,Normal,...,0,0,0,1,0,0,3,0,0,1
112,0,0,0,0,0,0,0,0,Normal,Normal,...,0,0,0,1,0,0,4,1,1,1
113,0,0,0,0,0,0,0,0,Normal,Bas,...,0,2,0,0,1,0,1,1,0,1
114,0,0,0,0,0,0,0,0,Eleve,Normal,...,0,1,0,0,1,0,5,0,0,0


In [ ]:
df_bayesian.head()

,Profil_ID,Alcool,Dépresseur,Stimulant,Hallucinogène,Anesthésique dissociatif,Narcotique analgésique,Inhalant,Pouls_cat,Temp_cat,...,Injection,Tonus musculaire,Yeux normaux,Injecté sang,Conjonctive rouge,Larmoyant,Test toucher,Perte equilibre,Depart hatif,Pivot incorrect
0,4,0,0,1,0,0,0,0,Eleve,Normal,...,0,0,0,1,0,0,0,1,1,1
1,1,1,0,0,0,0,0,0,Eleve,Normal,...,0,0,0,1,0,0,4,1,0,1
2,1,1,0,0,0,0,0,0,Eleve,Normal,...,0,0,0,0,1,0,4,0,0,1
3,1,1,0,0,0,0,0,0,Eleve,Normal,...,0,0,0,0,1,0,5,0,0,1
4,2,0,1,0,0,0,0,0,Normal,Normal,...,0,0,1,0,0,0,5,1,0,0


In [ ]:
df_final = ajuster_et_fusionner_final_expert(df_bayesian,dataset_augmented,cibles)


/tmp/ipykernel_703/270279701.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_r[c] = 0.5  # Padding neutre (Incertitude 50%)
/tmp/ipykernel_703/270279701.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_r[c] = 0.5  # Padding neutre (Incertitude 50%)
/tmp/ipykernel_703/270279701.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragm

In [ ]:
df_final.head()

,prob_0,prob_1,prob_2,prob_3,prob_4,prob_5,prob_6,prob_7,prob_8,prob_9,...,gaze,dt,charge,i_crash,f_t,f_vis,f_imu,f_gps,profil_id,target
0,2.067686e-09,6.817566e-04,1.008420e-15,1.241731e-03,4.724173e-01,1.881795e-10,5.040451e-17,2.219615e-29,9.021335e-08,2.220344e-10,...,0.2,0.972396,0.79882,0.0000,1.0,1.0,1.0,1.0,0.000246,4.0
1,4.385548e-01,5.755104e-17,2.994731e-36,1.212209e-04,2.378119e-03,5.204465e-24,1.114912e-08,2.458836e-13,3.318780e-14,5.858110e-14,...,0.2,1.630493,0.99755,0.7783,1.0,1.0,1.0,1.0,0.000015,0.0
2,2.654783e-01,2.230008e-01,9.204062e-11,2.693415e-15,2.067071e-01,1.378058e-09,7.691530e-05,5.523396e-03,3.048959e-27,1.368254e-07,...,0.2,22.410277,0.96638,0.0000,1.0,1.0,1.0,1.0,0.000292,0.0
3,1.605153e-16,8.956331e-04,3.821422e-01,1.626305e-05,2.530071e-03,2.272169e-11,1.840885e-05,4.236787e-36,8.747739e-08,2.146507e-14,...,0.2,1.906652,0.51651,0.4438,1.0,1.0,1.0,1.0,0.000062,2.0
4,2.598037e-01,1.040684e-06,1.687358e-19,3.136543e-01,6.033326e-61,2.217810e-16,1.168472e-10,1.594895e-15,5.801056e-05,4.977446e-07,...,0.2,9.681215,0.98548,0.0000,1.0,1.0,1.0,1.0,0.000138,0.0


In [ ]:
df_bayesian.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Profil_ID                 116 non-null    int64   
 1   Alcool                    116 non-null    int64   
 2   Dépresseur                116 non-null    int64   
 3   Stimulant                 116 non-null    int64   
 4   Hallucinogène             116 non-null    int64   
 5   Anesthésique dissociatif  116 non-null    int64   
 6   Narcotique analgésique    116 non-null    int64   
 7   Inhalant                  116 non-null    int64   
 8   Pouls_cat                 116 non-null    category
 9   Temp_cat                  116 non-null    category
 10  Tension_a_cat             116 non-null    category
 11  Tension_b_cat             116 non-null    category
 12  Ambiant_cat               116 non-null    category
 13  Noirceur_cat              116 non-null    category

In [ ]:
df_final.to_csv("dataset_final.csv", index=False)

In [ ]:
inference = VariableElimination(model_final)

r = inference.query(variables=['Profil_ID'], evidence={'Nistagmus horizontal_cat': 1, 'Pouls_cat': 'Normal', 'Convergence': 1})

In [ ]:
transformer_sample_en_probas_padded(r)

array([0.05211348, 0.12101305, 0.41290999, 0.18050103, 0.01018691,
       0.01106424, 0.14144616, 0.01304543, 0.00103008, 0.0023719 ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00179222, 0.        , 0.        , 0.0190836 ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.02760025, 0.        , 0.        ,
       0.        , 0.0023719 , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00346975, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     